# LXCat衝突断面積による電子swarm計算

LXCatの電子衝突断面積セットを `boltzpmp` へ読み込み、DC換算電場 $E/N$ に対する
EEDF、平均電子エネルギー、ドリフト速度、換算移動度、電離周波数、反応速度係数を計算します。

このリポジトリの `xsec/helium ion Cross section.txt` と
`xsec/argon ion Cross section.txt` も実際に読み込んで形式を確認します。ただし、これらは
`He+ + He` / `Ar+ + Ar` の**イオン中性粒子衝突断面積**であり、電子swarmソルバーの入力ではありません。
そのため既定の計算には、同梱の電子衝突セット
`examples/data/Ar_IST-Lisbon_LXCat.txt` を使用します。

Heの電子swarmを計算する場合は、LXCatから `ELASTIC` または `EFFECTIVE` と、必要な
`EXCITATION` / `IONIZATION` などを含む**電子-He衝突セット**を用意し、次のパラメータセルで
`ELECTRON_LXCAT_PATH`、`GAS_NAME`、`GAS_MASS_AMU` を変更してください。


In [ ]:
import math
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np

import boltzpmp as bp

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("matplotlibがないためグラフ描画をスキップします")

warnings.filterwarnings("ignore", message="EEPF at eps_max.*")


def find_repo_root(start=Path.cwd()):
    """リポジトリ直下とexamples直下のどちらから起動してもルートを見つける。"""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "python" / "boltzpmp").is_dir():
            return candidate
    raise FileNotFoundError("boltzpmpのリポジトリルートが見つかりません")


ROOT = find_repo_root()
print("repository root:", ROOT)


In [ ]:
# ============ 入力と計算条件 ============
# 指定されたxsecファイルは下のセルで読み込み、電子衝突セットかどうかを判定する。
INSPECTION_PATHS = [
    ROOT / "xsec" / "helium ion Cross section.txt",
    ROOT / "xsec" / "argon ion Cross section.txt",
]

# 実際の電子swarm計算に使うLXCat電子衝突セット。
ELECTRON_LXCAT_PATH = ROOT / "examples" / "data" / "Ar_IST-Lisbon_LXCat.txt"
GAS_NAME = "Ar"
GAS_MASS_AMU = 39.948

FIELDS_TD = np.array([10.0, 50.0, 100.0, 500.0])
PRESSURE_PA = 133.0
GAS_TEMPERATURE_K = 273.0

N_CELLS = 240
N_THETA = 48
TOL = 1e-5
MAX_STEPS = 2_000_000
MAX_RETRIES = 8
TAIL_MAX = 1e-6
TAIL_MIN = 1e-12

print("electron LXCat input:", ELECTRON_LXCAT_PATH.relative_to(ROOT))
print("E/N [Td]:", FIELDS_TD)


In [ ]:
# ============ 指定xsecファイルの読み込みと種別確認 ============
ELECTRON_KINDS = {"ELASTIC", "EFFECTIVE", "EXCITATION", "IONIZATION", "ATTACHMENT"}


def inspect_lxcat_file(path):
    """ファイルを読み、電子LXCatブロックとイオン衝突メタデータを要約する。"""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"断面積ファイルが見つかりません: {path}")
    text = path.read_text(encoding="utf-8-sig")
    lines = text.splitlines()
    electron_kinds = [line.strip() for line in lines if line.strip() in ELECTRON_KINDS]
    species = [line.split(":", 1)[1].strip() for line in lines if line.upper().startswith("SPECIES:")]
    processes = [line.split(":", 1)[1].strip() for line in lines if line.upper().startswith("PROCESS:")]
    parsed = bp.parse_lxcat(path)
    return {
        "path": path,
        "bytes": path.stat().st_size,
        "electron_kinds": Counter(electron_kinds),
        "parsed_electron_processes": len(parsed),
        "species": species,
        "processes": processes,
    }


for path in INSPECTION_PATHS:
    if not path.is_file():
        print(f"\n[{path.relative_to(ROOT)}] not found; inspection skipped")
        continue
    info = inspect_lxcat_file(path)
    print(f"\n[{info['path'].relative_to(ROOT)}] {info['bytes']:,} bytes")
    print("  electron blocks:", dict(info["electron_kinds"]) or "none")
    print("  parsed electron processes:", info["parsed_electron_processes"])
    print("  species metadata:", info["species"] or "none")
    print("  process metadata:", info["processes"] or "none")
    if not info["parsed_electron_processes"]:
        print("  verdict: イオン輸送用データのため電子swarm入力には使用しない")


In [ ]:
# ============ 電子LXCatセットの読み込みと検証 ============
def load_electron_lxcat(path):
    """電子swarmに必要な弾性過程を含むLXCatセットを検証して返す。"""
    path = Path(path).resolve(strict=True)
    cross_sections = bp.parse_lxcat(path)
    if not cross_sections:
        raise ValueError(
            f"{path.name} に電子衝突ブロックがありません。"
            "ELASTIC/EFFECTIVE等を含む電子衝突LXCatセットを指定してください。"
        )
    if not any(cs.kind in {"ELASTIC", "EFFECTIVE"} for cs in cross_sections):
        raise ValueError("電子swarm計算にはELASTICまたはEFFECTIVE断面積が必要です")

    for cs in cross_sections:
        data = np.asarray(cs.data, dtype=float)
        if data.shape[0] < 2 or not np.all(np.isfinite(data)):
            raise ValueError(f"{cs.name}: 断面積表が空、または非有限値を含みます")
        if np.any(np.diff(data[:, 0]) <= 0.0):
            raise ValueError(f"{cs.name}: エネルギー列が単調増加ではありません")
        if np.any(data[:, 0] < 0.0) or np.any(data[:, 1] < 0.0):
            raise ValueError(f"{cs.name}: 負のエネルギーまたは断面積を含みます")
    return cross_sections


cross_sections = load_electron_lxcat(ELECTRON_LXCAT_PATH)
kind_counts = Counter(cs.kind for cs in cross_sections)
mixture = bp.Mixture(
    [bp.Gas(GAS_NAME, 1.0, cross_sections, mass_amu=GAS_MASS_AMU)],
    p_Pa=PRESSURE_PA,
    T_K=GAS_TEMPERATURE_K,
)

print(f"loaded {len(cross_sections)} electron collision processes: {dict(kind_counts)}")
print(f"gas number density: {mixture.N:.6e} m^-3")
print(f"energy coverage: 0 .. {max(cs.data[-1, 0] for cs in cross_sections):g} eV")
for i, cs in enumerate(cross_sections, start=1):
    print(f"{i:2d}. {cs.kind:10s} threshold={cs.threshold:7.3f} eV  {cs.name}")


In [ ]:
# ============ 読み込んだ断面積の可視化 ============
if plt is None:
    print("断面積グラフを描くにはmatplotlibを利用できる環境で実行してください")
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    for cs in cross_sections:
        data = np.asarray(cs.data, dtype=float)
        label = f"{cs.kind}: {cs.name}"
        ax.loglog(np.maximum(data[:, 0], 1e-4), np.maximum(data[:, 1], 1e-30), label=label)
    ax.set_xlabel("electron energy [eV]")
    ax.set_ylabel("cross section [m$^2$]")
    ax.set_title(f"Electron collision cross sections: {GAS_NAME}")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============ EEPF末端を確認するeps_max自動調整ソルバ ============
def eps_max_guess(EN_Td, processes):
    """既存の広域Arスイープ式と最大しきい値から初期上限を決める。"""
    x = math.log10(float(EN_Td))
    empirical = 1.3 * 10 ** (0.1805 * x * x - 0.0346 * x + 0.9967)
    max_threshold = max((float(cs.threshold) for cs in processes), default=0.0)
    return max(20.0, empirical, 1.25 * max_threshold)


def solve_adaptive(mixture, processes, EN_Td, *, n_cells=N_CELLS, n_theta=N_THETA,
                   tol=TOL, max_steps=MAX_STEPS, max_retries=MAX_RETRIES):
    """EEPF末端比が許容範囲に入るまでエネルギー上限を調整する。"""
    eps_max = eps_max_guess(EN_Td, processes)
    shrunk = False
    for attempt in range(1, max_retries + 1):
        solver = bp.PMSolver(
            mixture,
            eps_max_eV=eps_max,
            d_eps_eV=eps_max / n_cells,
            n_theta=n_theta,
        )
        result = solver.solve_dc(
            EN_Td=float(EN_Td),
            tol=tol,
            max_steps=max_steps,
            check_every=200,
        )
        if not result.converged:
            raise RuntimeError(f"{EN_Td:g} Td: {result.n_steps}ステップで未収束")

        tail = float(result.extra["eepf_tail_ratio"])
        if tail > TAIL_MAX:
            eps_max *= 1.6
            continue
        if tail < TAIL_MIN and not shrunk:
            eepf = np.asarray(result.eepf, dtype=float)
            above_floor = np.flatnonzero(eepf / max(eepf.max(), 1e-300) > 1e-8)
            if above_floor.size:
                last_energy = float(np.asarray(result.energy_grid)[above_floor[-1]])
                reduced_eps_max = max(5.0, 1.3 * last_energy, 1.25 * max(cs.threshold for cs in processes))
                if reduced_eps_max < 0.7 * eps_max:
                    eps_max = reduced_eps_max
                    shrunk = True
                    continue
        return result, eps_max, attempt
    raise RuntimeError(f"{EN_Td:g} Td: eps_max自動調整が{max_retries}回で完了しませんでした")


In [ ]:
# ============ 電子swarmスイープ実行 ============
results = []
for EN_Td in np.sort(FIELDS_TD):
    started = time.perf_counter()
    result, eps_max, attempts = solve_adaptive(mixture, cross_sections, float(EN_Td))
    elapsed = time.perf_counter() - started
    results.append(result)
    reduced_mobility = result.drift_velocity / (float(EN_Td) * 1e-21)
    print(
        f"E/N={EN_Td:7.1f} Td  <e>={result.mean_energy:8.4f} eV  "
        f"vd={result.drift_velocity:10.3e} m/s  muN={reduced_mobility:10.3e} 1/(V m s)  "
        f"nu_i/N={result.reduced_ionization_frequency:10.3e} m3/s  "
        f"eps_max={eps_max:7.1f} eV  steps={result.n_steps:7d}  "
        f"retries={attempts}  {elapsed:.2f}s"
    )


In [ ]:
# ============ EEDFとswarmパラメータの可視化 ============
fields = np.array([result.extra["EN_Td"] for result in results], dtype=float)
mean_energies = np.array([result.mean_energy for result in results], dtype=float)
drift_velocities = np.array([result.drift_velocity for result in results], dtype=float)
reduced_mobilities = drift_velocities / (fields * 1e-21)
reduced_ionization = np.array([result.reduced_ionization_frequency for result in results], dtype=float)

if plt is None:
    print("swarmグラフを描くにはmatplotlibを利用できる環境で実行してください")
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    for result in results:
        axes[0, 0].loglog(result.energy_grid, result.eepf, label=f"{result.extra['EN_Td']:g} Td")
    axes[0, 0].set_xlabel("electron energy [eV]")
    axes[0, 0].set_ylabel("EEPF [eV$^{-3/2}$]")
    axes[0, 0].set_title(f"EEPF: e + {GAS_NAME}")
    axes[0, 0].legend()

    axes[0, 1].loglog(fields, mean_energies, "o-")
    axes[0, 1].set_xlabel("E/N [Td]")
    axes[0, 1].set_ylabel("mean electron energy [eV]")
    axes[0, 1].set_title("Mean energy")

    axes[1, 0].loglog(fields, reduced_mobilities, "s-", color="tab:red")
    axes[1, 0].set_xlabel("E/N [Td]")
    axes[1, 0].set_ylabel("reduced mobility $\\mu N$ [1/(V m s)]")
    axes[1, 0].set_title("Reduced electron mobility")

    positive = reduced_ionization > 0.0
    if np.any(positive):
        axes[1, 1].loglog(fields[positive], reduced_ionization[positive], "^-", color="tab:green")
    axes[1, 1].set_xlabel("E/N [Td]")
    axes[1, 1].set_ylabel("reduced ionization frequency [m$^3$/s]")
    axes[1, 1].set_title("Ionization")

    for ax in axes.flat:
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============ 反応速度係数 ============
reaction_names = sorted({name for result in results for name in result.rate_coefficients})
print("E/N [Td]	process	rate coefficient [m3/s]")
for result in results:
    EN_Td = float(result.extra["EN_Td"])
    for name in reaction_names:
        value = float(result.rate_coefficients.get(name, 0.0))
        print(f"{EN_Td:g}\t{name}\t{value:.6e}")


## 入力セットを変更するときの確認事項

1. `ELECTRON_LXCAT_PATH` を電子衝突LXCatファイルへ変更する。
2. `GAS_NAME` と `GAS_MASS_AMU` を対象ガスへ合わせる（Heなら `He` と `4.002602`）。
3. 少なくとも `ELASTIC` または `EFFECTIVE` ブロックが必要。励起、電離、付着を評価する場合は
   対応するブロックも含める。
4. 断面積の最大エネルギーよりEEDFの裾が大きく伸びる条件では、最終断面積値による外挿の影響を受ける。
   高いE/Nを使う前にエネルギーカバー範囲を確認する。
5. 低いE/Nでは緩和が遅くなるため、まず10 Td以上で動作確認し、その後 `FIELDS_TD` を広げる。

`xsec/helium ion Cross section.txt` のような `SPECIES: He^+ / He`、
`PROCESS: He+ + He ...` 形式はイオンswarm用です。現在の `boltzpmp.PMSolver` は電子用なので、
この種のファイルを電子断面積へ読み替えて使用しないでください。
